# Gabarito — Módulo 2: Preparing Text Data

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

sms = pd.read_csv("sms_clean.csv")
with open("vocab.json") as f:
    word2idx = json.load(f)
embedding_matrix = np.load("embedding_matrix.npy")

print(sms.shape, len(word2idx), embedding_matrix.shape)
sms.head()

(100, 3) 225 (225, 32)


,label,text,clean_text
0,ham,"Hey, are we still on for lunch tomorrow?",hey are we still on for lunch tomorrow
1,ham,I'll call you when I get home from work.,ill call you when i get home from work
2,ham,Can you send me the notes from today's class?,can you send me the notes from todays class
3,ham,Happy birthday! Hope you have an amazing day.,happy birthday hope you have an amazing day
4,ham,"Running a bit late, be there in 10 minutes.",running a bit late be there in minutes


In [2]:
# 2.1
print(sms["clean_text"].isnull().sum())
sms = sms.dropna(subset=["clean_text"])

0


In [3]:
# 2.2
y = sms["label"].map({"ham": 0, "spam": 1})

unk = word2idx["<UNK>"]
X_seq = [
    [word2idx.get(w, unk) for w in text.split()]
    for text in sms["clean_text"]
]
print(X_seq[0], y.iloc[0])

[91, 16, 209, 182, 143, 72, 114, 197] 0


In [4]:
# 2.3
X_train_seq, X_test_seq, y_train, y_test = train_test_split(
    X_seq, y, test_size=0.25, random_state=1, stratify=y
)
print(len(X_train_seq), len(X_test_seq))

75 25


In [5]:
# 2.4
maxlen = 15

def pad_sequences(seqs, maxlen):
    padded = np.zeros((len(seqs), maxlen), dtype=np.int64)
    for i, seq in enumerate(seqs):
        trimmed = seq[:maxlen]
        padded[i, :len(trimmed)] = trimmed
    return padded

X_train_pad = pad_sequences(X_train_seq, maxlen)
X_test_pad = pad_sequences(X_test_seq, maxlen)
print(X_train_pad.shape, X_test_pad.shape)

(75, 15) (25, 15)


In [6]:
# 2.5
np.save("X_train_pad.npy", X_train_pad)
np.save("X_test_pad.npy", X_test_pad)
np.save("y_train.npy", y_train.values)
np.save("y_test.npy", y_test.values)

config = {
    "maxlen": maxlen,
    "vocab_size": len(word2idx),
    "embed_dim": embedding_matrix.shape[1],
}
with open("config.json", "w") as f:
    json.dump(config, f)

test_text = sms.loc[y_test.index, ["clean_text", "label"]]
test_text.to_csv("test_text.csv", index=False)

print("Salvo: X_train_pad.npy, X_test_pad.npy, y_train.npy, y_test.npy, config.json, test_text.csv")

Salvo: X_train_pad.npy, X_test_pad.npy, y_train.npy, y_test.npy, config.json, test_text.csv
